# Setup

In [ ]:
!pip install -qU openai google-api-python-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.4/567.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 16.0 MB/s eta 0:00:00


- OpenAI - narzędzia do pracy z API OpenAI.

- google-api-python-client - oficjalny klient API Google dla Pythona, który ułatwia integrację z usługami Google jak Gmail, Drive, Sheets czy YouTube.

In [ ]:
from openai import OpenAI
from googleapiclient.discovery import build
import re
import time
from pprint import pp
from google.colab import userdata

In [ ]:
class CFG:
    model = "gpt-4o-mini"
    max_tokens = 2048

In [ ]:
api_key = userdata.get("openaivision")
client = OpenAI(api_key=api_key)

In [ ]:
# setup the search engine
cse_key = userdata.get("google_cse")
cse_id = userdata.get("cse_id")

Ten fragment kodu zajmuje się konfiguracją dostępu do wyszukiwarki Google, która będzie wykorzystywana w dalszej części programu do wyszukiwania informacji w internecie.

Setup CSE: https://programmablesearchengine.google.com/controlpanel/all

# Funkcje


In [ ]:
def search(search_term):
    search_result = ""
    service = build("customsearch", "v1", developerKey=cse_key)
    res = service.cse().list(q=search_term, cx=cse_id, num=10).execute()
    for result in res["items"]:
        search_result = search_result + result["snippet"]

    return search_result

Ta funkcja `search` została zaprojektowana do wyszukiwania informacji w internecie za pomocą Google Custom Search Engine (CSE). Przyjrzyjmy się jej działaniu krok po kroku:

Funkcja przyjmuje jeden parametr `search_term`, który jest zapytaniem, czyli frazą, którą chcemy wyszukać w internecie.

Na początku tworzona jest pusta zmienna `search_result`, która będzie stopniowo wypełniana wynikami wyszukiwania.

Następnie tworzony jest obiekt usługi wyszukiwania Google poprzez wywołanie funkcji `build` z trzema parametrami:
- "customsearch" - nazwa usługi, którą chcemy wykorzystać
- "v1" - wersja API
- `developerKey = cse_key` - klucz API pobrany wcześniej z magazynu userdata

Po utworzeniu obiektu usługi, wykonywane jest właściwe zapytanie do API za pomocą metody `service.cse().list()`, której przekazywane są parametry:
- `q = search_term` - zapytanie wyszukiwania (query)
- `cx = cse_id` - identyfikator niestandardowej wyszukiwarki
- `num = 10` - liczba wyników do pobrania

Metoda `.execute()` wysyła zapytanie do serwerów Google i odbiera wyniki.

Następnie kod przechodzi przez listę wyników (zawartą w `res['items']`) i dla każdego wyniku wyszukiwania dodaje jego skrót (`snippet`) do zmiennej `search_result`. Skrót to krótki fragment tekstu ze strony internetowej, który zawiera kontekst wyszukiwanego terminu.

Na końcu funkcja zwraca połączony tekst wszystkich skrótów jako jeden ciąg znaków.

In [ ]:
def generate_answer(prompt):
    response = client.chat.completions.create(
        model=CFG.model,
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant that writes essays.",
            },
            {"role": "user", "content": prompt},
        ],
        max_tokens=CFG.max_tokens,
        n=1,
        stop=None,
    )

    essay = response.choices[0].message.content.strip()
    pp(essay)

In [ ]:
def extract_action_and_input(text):
    action_pattern = r"Action: (.+?)\n"
    input_pattern = r"Action Input: \"(.+?)\""
    action = re.findall(action_pattern, text)
    action_input = re.findall(input_pattern, text)
    return action, action_input

In [ ]:
# update the system prompt
system_prompt2 = """
Answer the following questions and obey the following commands as best you can.

You have access to the following tools:

Search: Search: useful for when you need to answer questions about current events. You should ask targeted questions.
Response To Human: When you need to respond to the human you are talking to.

You will receive a message from the human, then you should start a loop and do one of two things

Option 1: You use a tool to answer the question.
For this, you should use the following format:
Thought: you should always think about what to do
Action: the action to take, should be one of [Search]
Action Input: "the input to the action, to be sent to the tool"

Option 2: You respond to the human.
For this, you should use the following format:
Action: Response To Human
Action Input: "your response to the human, summarizing what you did and what you learned"

Begin!
"""

In [ ]:
def agentic_answer(prompt):
    messages = [
        {"role": "system", "content": system_prompt2},
        {"role": "user", "content": prompt},
    ]

    while True:
        response = client.chat.completions.create(
            model=CFG.model,
            messages=messages,
            temperature=0,
            top_p=1,
        )
        response_text = response.choices[0].message.content
        # enforce a wait to prevent the Rate Limit error for free-tier users
        time.sleep(3)

        action, action_input = extract_action_and_input(response_text)
        if action[-1] == "Search":
            tool = search
        elif action[-1] == "Response To Human":
            pp(f"Response: {action_input[-1]}")
            break
        observation = tool(action_input[-1])
        messages.extend(
            [
                {"role": "system", "content": response_text},
                {"role": "user", "content": f"Observation: {observation}"},
            ]
        )

Ta funkcja `agentic_answer` implementuje agenta AI, który może wykonywać działania w odpowiedzi na zapytanie użytkownika. Jest to przykład tzw. pętli ReAct (Reasoning and Acting), gdzie model językowy na przemian rozumuje i podejmuje działania, aby lepiej odpowiedzieć na złożone pytania.

Funkcja przyjmuje jeden parametr `prompt` - zapytanie użytkownika, na które agent ma odpowiedzieć.

Na początku tworzony jest początkowy zestaw wiadomości składający się z:
- Wiadomości systemowej zawierającej instrukcje dla modelu (zdefiniowanej w zmiennej `system_prompt2`)
- Wiadomości użytkownika zawierającej przekazane zapytanie

Następnie rozpoczyna się główna pętla funkcji `while True`, która będzie wykonywana do momentu, gdy agent zdecyduje, że ma już ostateczną odpowiedź.

W każdej iteracji pętli:
1. Wysyłane jest zapytanie do modelu OpenAI z aktualnymi wiadomościami, przy czym `temperature=0` oznacza, że model będzie generował bardziej deterministyczne, mniej losowe odpowiedzi. Parameter `top_p=1` pozwala modelowi na rozważenie wszystkich możliwych tokenów podczas generowania odpowiedzi.

2. Pobierany jest wygenerowany tekst odpowiedzi.

3. Wprowadzane jest opóźnienie 3 sekundy (`time.sleep(3)`), aby uniknąć przekroczenia limitu żądań dla użytkowników korzystających z darmowego planu.

4. Z odpowiedzi modelu wyodrębniane są akcja i dane wejściowe przy użyciu wcześniej zdefiniowanej funkcji `extract_action_and_input`.

5. Na podstawie wyodrębnionej akcji podejmowana jest decyzja:
   - Jeśli akcja to "Search", agent będzie używał funkcji `search` do wyszukiwania informacji w internecie.
   - Jeśli akcja to "Response To Human", agent wyświetli końcową odpowiedź i zakończy pętlę przy użyciu instrukcji `break`.

6. Jeśli wykonywana jest akcja (np. wyszukiwanie), jej wynik jest zapisywany jako `observation`.

7. Do listy wiadomości dodawane są:
   - Odpowiedź modelu jako wiadomość systemowa
   - Wynik obserwacji (np. wyniki wyszukiwania) jako wiadomość użytkownika

Ta pętla pozwala agentowi na:
- Analizowanie problemu i decydowanie, jakie informacje są potrzebne
- Wyszukiwanie tych informacji w internecie
- Analizowanie znalezionych informacji
- Szukanie dodatkowych informacji, jeśli to konieczne
- Ostatecznie formułowanie odpowiedzi, która uwzględnia wszystkie zebrane dane

Jest to zaawansowana implementacja agenta AI, który może samodzielnie poszukiwać informacji, aby udzielić jak najlepszej odpowiedzi na pytanie użytkownika, zamiast polegać wyłącznie na swojej wbudowanej wiedzy.

# Model po prostu

In [ ]:
muhprompt = "Who is Konrad Banachewicz"

In [ ]:
generate_answer(muhprompt)

('As of my last knowledge update in October 2021, Konrad Banachewicz is not a '
 'widely recognized public figure, and there may not be notable information or '
 'a well-documented biography available about him. If he has become prominent '
 'or relevant in a specific context since then, I would not have that updated '
 'information. \n'
 '\n'
 'If you are looking for information about Konrad Banachewicz in a specific '
 'context, such as in academia, sports, or another field, please provide more '
 'details, and I will do my best to assist you!')


# Model z narzędziem

In [ ]:
agentic_answer(muhprompt)

('Response: Konrad Banachewicz is a data science manager with extensive '
 'experience in the field. He holds a PhD in statistics from Vrije '
 'Universiteit and has been involved in various projects related to data '
 'interrogation and analysis. He is currently associated with Adevinta and has '
 'a background in time-series analysis, as evidenced by his contributions to '
 'educational content in that area.')
